<a href="https://colab.research.google.com/github/Ayush-Singh-36/Ayush-Singh-36/blob/main/transcribing_model_pytorch_training_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Development

# Device agnostic code

In [22]:
import torch

if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

# Downloading the dataset directly from kaggle

In [23]:
import os
from google.colab import userdata
import sys
import pandas as pd
def custom_exit(status):
    print(f"Kaggle API tried to exit with status {status}. Ignoring for Colab environment.")
    # Optionally, raise an exception or log, instead of actual exit.
sys.exit = custom_exit
exit = custom_exit # Patch the global 'exit' function as well
try:
    __builtins__.exit = custom_exit # Explicitly patch built-in exit
except AttributeError:
    print("Could not patch __builtins__.exit - it might not be present or modifiable in this environment.")

# Retrieve credentials from Colab secrets
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# Now import and run the download code
from kaggle.api.kaggle_api_extended import KaggleApi

dataset_slug = "mozillaorg/common-voice"
download_path = "./data"

print("Authenticating via environment variables...")
api = KaggleApi()
api.authenticate()

print("Downloading movie dataset from Kaggle...")
api.dataset_download_files(dataset_slug, path=download_path, unzip=True)

print(f"Done! Your files have been saved to the '{download_path}' folder.")

Authenticating via environment variables...
Dataset URL: https://www.kaggle.com/datasets/mozillaorg/common-voice
Done! Your files have been saved to the './data' folder.


# Unzipping rest files

In [24]:
import zipfile
import os
zip_file_to_unzip = "/content/data/common-voice.zip"

# Define the extraction path. Let's create a new folder for these files to avoid clutter.
extraction_folder = "./data/common-voice-unzipped"
os.makedirs(extraction_folder, exist_ok=True)

print(f"Unzipping '{zip_file_to_unzip}' to '{extraction_folder}'...")
with zipfile.ZipFile(zip_file_to_unzip, 'r') as zip_ref:
    zip_ref.extractall(extraction_folder)

print("Unzipping complete!")

Unzipping '/content/data/common-voice.zip' to './data/common-voice-unzipped'...
Unzipping complete!


# Looking at info of the dataset

In [25]:
# List the contents of the 'data' folder
print(f"Contents of the '{download_path}' folder:")
for root, dirs, files in os.walk(download_path):
    for name in dirs:
        print(os.path.join(root, name))

Contents of the './data' folder:
./data/common-voice-unzipped
./data/common-voice-unzipped/cv-other-dev
./data/common-voice-unzipped/cv-valid-dev
./data/common-voice-unzipped/cv-invalid
./data/common-voice-unzipped/cv-valid-train
./data/common-voice-unzipped/cv-valid-test
./data/common-voice-unzipped/cv-other-test
./data/common-voice-unzipped/cv-other-train
./data/common-voice-unzipped/cv-other-dev/cv-other-dev
./data/common-voice-unzipped/cv-valid-dev/cv-valid-dev
./data/common-voice-unzipped/cv-invalid/cv-invalid
./data/common-voice-unzipped/cv-valid-train/cv-valid-train
./data/common-voice-unzipped/cv-valid-test/cv-valid-test
./data/common-voice-unzipped/cv-other-test/cv-other-test
./data/common-voice-unzipped/cv-other-train/cv-other-train


# Keeping useful files and folders only

In [26]:
import os
import shutil

data_directory = './data'
excluded_files = ['common-voice.zip']
excluded_folders = ['common-voice-unzipped']

print(f"Cleaning up directory: {data_directory}")

for item in os.listdir(data_directory):
    item_path = os.path.join(data_directory, item)
    if os.path.isfile(item_path):
        if item not in excluded_files:
            os.remove(item_path)
            print(f"Removed file: {item_path}")
    elif os.path.isdir(item_path):
        if item not in excluded_folders:
            shutil.rmtree(item_path)
            print(f"Removed directory: {item_path}")

print("Cleanup complete.")

Cleaning up directory: ./data
Cleanup complete.


# Looking at dataset once again

In [27]:
# List the contents of the 'data' folder
print(f"Contents of the '{download_path}' folder:")
for root, dirs, files in os.walk(download_path):
    for name in dirs:
        print(os.path.join(root, name))

Contents of the './data' folder:
./data/common-voice-unzipped
./data/common-voice-unzipped/cv-other-dev
./data/common-voice-unzipped/cv-valid-dev
./data/common-voice-unzipped/cv-invalid
./data/common-voice-unzipped/cv-valid-train
./data/common-voice-unzipped/cv-valid-test
./data/common-voice-unzipped/cv-other-test
./data/common-voice-unzipped/cv-other-train
./data/common-voice-unzipped/cv-other-dev/cv-other-dev
./data/common-voice-unzipped/cv-valid-dev/cv-valid-dev
./data/common-voice-unzipped/cv-invalid/cv-invalid
./data/common-voice-unzipped/cv-valid-train/cv-valid-train
./data/common-voice-unzipped/cv-valid-test/cv-valid-test
./data/common-voice-unzipped/cv-other-test/cv-other-test
./data/common-voice-unzipped/cv-other-train/cv-other-train


# Checking the cardinality of the dataset

In [28]:
import os

# The root directory for the unzipped common-voice data
unzipped_data_root = "./data/common-voice-unzipped"

file_count = 0
directory_count = 0

if os.path.exists(unzipped_data_root):
    print(f"Checking cardinality for: {unzipped_data_root}")
    for root, dirs, files in os.walk(unzipped_data_root):
        directory_count += len(dirs)
        file_count += len(files)

    print(f"Total unique files: {file_count}")
    print(f"Total unique directories (including subdirectories): {directory_count}")
else:
    print(f"Error: The directory '{unzipped_data_root}' does not exist.")

Checking cardinality for: ./data/common-voice-unzipped
Total unique files: 380377
Total unique directories (including subdirectories): 14
